# Orchestrator

Примеры использования пайплайна:

In [15]:
import sys
import os
import json

from app.orchestrator.runner import run
from app.models.state import PipelineResult

# Добавляем корень репозитория в путь, чтобы импортировать пакет app
sys.path.insert(0, os.path.abspath(".."))

In [16]:
# Можно задать переменные окружения в .env, либо перезаписать тут.
os.environ["MODEL_NAME"] = "openai/gpt-4o-mini"

## Запуск пайплана целиком

In [14]:
query = "Show me all users who registered in the last 30 days"

result: PipelineResult = run(query)
result

PipelineResult(final_sql="SELECT id, username, email FROM users WHERE registration_date >= NOW() - INTERVAL '30 days' LIMIT 100", approved=True, iterations_used=2, iteration_logs=[IterationLog(iteration=0, sql_query="SELECT id, username, email, registration_date FROM users WHERE registration_date >= NOW() - INTERVAL '30 days'", judge_output=JudgeOutput(verdict=<Verdict.REJECTED: 'REJECTED'>, risk_score=5.0, issues=[FoundIssue(type=<IssueType.SECURITY: 'SECURITY'>, severity=<Severity.MEDIUM: 'MEDIUM'>, message='SELECT * exposes all columns including sensitive ones', fix_instruction='Specify only the needed columns in the SELECT statement instead of using SELECT *.'), FoundIssue(type=<IssueType.SECURITY: 'SECURITY'>, severity=<Severity.MEDIUM: 'MEDIUM'>, message='Direct access to sensitive columns (passwords, tokens, PII)', fix_instruction='Review and avoid querying sensitive data fields directly, and ensure sensitive data is adequately protected.'), FoundIssue(type=<IssueType.PERFORMANC

In [ ]:
print(f"Approved: {result.approved}")
print(f"Iterations used: {result.iterations_used}")
print(f"\nFinal SQL:\n{result.final_sql}")

print("\nIteration logs:")
print(json.dumps([log.model_dump(mode="json") for log in result.iteration_logs], indent=2))

Approved: False
Iterations used: 3

Final SQL:
SELECT username, registration_date FROM users WHERE registration_date >= NOW() - INTERVAL '30 days'

Iteration logs:
[
  {
    "iteration": 0,
    "sql_query": "SELECT username, registration_date FROM users WHERE registration_date >= NOW() - INTERVAL '30 days'",
    "judge_output": {
      "verdict": "REJECTED",
      "risk_score": 6.0,
      "issues": [
        {
          "type": "SECURITY",
          "severity": "MEDIUM",
          "message": "SELECT * exposes all columns including sensitive ones.",
          "fix_instruction": "Specify only the columns needed. Example: SELECT username, registration_date FROM users."
        },
        {
          "type": "SECURITY",
          "severity": "LOW",
          "message": "Direct access to sensitive columns potentially exposing PII.",
          "fix_instruction": "Ensure that user data complies with privacy regulations and consider using views to limit access."
        },
        {
          

## Запуск Generator

In [11]:
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

from app.config import get_config
from app.generator.agent import GeneratorAgent

In [12]:
config = get_config()
llm = ChatOpenAI(
    base_url=config.openrouter_base_url,
    api_key=config.open_router_api_key,
    model=config.model_name,
)

generator = GeneratorAgent(llm)

state = {
    "messages": [HumanMessage(content="Show me all users who registered in the last 30 days")],
    "judge_responses": [],
    "llm_calls": 0,
    "audit_iteration": 0,
}

output = generator(state)
print(output)

{'messages': [AIMessage(content='{\n  "sql": "SELECT user_id, username, registered_date FROM users WHERE registered_date >= NOW() - INTERVAL \'30 days\'",\n  "tables_used": [\n    "users"\n  ]\n}', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'llm_calls': 1}


Других агентов можно запускать по аналогии.